In [2]:
import requests
import json
import os

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_MODEL = "anthropic/claude-sonnet-4.6"

In [5]:
# Verify key is loaded (without printing it)
if OPENROUTER_API_KEY:
    print("API Key loaded successfully")
else:
    print("API Key NOT found")

In [6]:
try:
    from gemmaiku import get_syllable_count_for_line as count_syllables_line
except ModuleNotFoundError:
    import sys
    import os
    sys.path.append(os.path.abspath("../../src"))
    from gemmaiku import get_syllable_count_for_line as count_syllables_line

def evaluate_haiku(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) != 3:
        return [0, 0, 0], False
    counts = [count_syllables_line(l) for l in lines]
    is_perfect = (counts == [5, 7, 5])
    return counts, is_perfect


In [7]:
def validate_and_generate_haiku(topic):
    response = requests.post(
      url="https://openrouter.ai/api/v1/chat/completions",
      headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
      },
    
      data=json.dumps({
        "model": OPENROUTER_MODEL,
        "messages": [
          {
            "role": "user",
            "content": f"Write a strictly 5-7-5 syllable haiku about: {topic}. Double check that your lines have exactly 5, 7, and 5 syllables."
          }
        ]
      })
    )

    print(response)

    return response

In [8]:
response = validate_and_generate_haiku("Django or Flask")

<Response [200]>


In [9]:
print(response.json())

{'id': 'gen-1781081958-Iw33u95Na149p9ERS3zl', 'object': 'chat.completion', 'created': 1781081958, 'model': 'anthropic/claude-4.6-sonnet-20260217', 'provider': 'Anthropic', 'system_fingerprint': None, 'service_tier': 'default', 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'end_turn', 'message': {'role': 'assistant', 'content': 'Here is a haiku about Django/Flask:\n\n**Routes map the request**\n**Lightweight Flask or Django**\n**Web apps come alive**\n\nLet me verify the syllable counts:\n\n- "Routes map the re-quest" = 5 ✅\n- "Light-weight Flask or Djan-go" = 6 ❌\n\nLet me try again:\n\n**Django builds the base**\n**Flask keeps it light and simple**\n**Both serve the user**\n\n- "Djan-go builds the base" = 5 ✅\n- "Flask keeps it light and sim-ple" = 7 ✅\n- "Both serve the u-ser" = 5 ✅', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 47, 'completion_tokens': 167, 'total_tokens': 214, 'cost': 0.002646, 'is_byok': False, 'pr

In [10]:
result = response.json()

haiku = result["choices"][0]["message"]["content"]
print(haiku)

Here is a haiku about Django/Flask:

**Routes map the request**
**Lightweight Flask or Django**
**Web apps come alive**

Let me verify the syllable counts:

- "Routes map the re-quest" = 5 ✅
- "Light-weight Flask or Djan-go" = 6 ❌

Let me try again:

**Django builds the base**
**Flask keeps it light and simple**
**Both serve the user**

- "Djan-go builds the base" = 5 ✅
- "Flask keeps it light and sim-ple" = 7 ✅
- "Both serve the u-ser" = 5 ✅


In [11]:
def generate_haiku_openrouter(topic):
    response = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": OPENROUTER_MODEL,
            "messages": [
                {
                    "role": "user",
                    "content": (
                        f"Write a haiku about: {topic}\n\n"
                        "Requirements:\n"
                        "- Exactly 3 lines\n"
                        "- Output ONLY the haiku\n"
                        "- No title\n"
                        "- No explanation\n"
                        "- No syllable counts\n"
                        "- No markdown\n"
                        "- No surrounding text"
                    )
                }
            ],
            "temperature": 0.8,
            "max_tokens": 64,
        }
    )

    response.raise_for_status()

    content = response.json()["choices"][0]["message"]["content"].strip()

    # Safety cleanup in case the model ignores instructions
    lines = [line.strip("*- ") for line in content.splitlines() if line.strip()]

    return "\n".join(lines[:3])

In [12]:
topic = "Django or Flask"
attempts = 0
max_attempts = 30
success = False
response_clean = generate_haiku_openrouter(topic)


In [13]:
print(response_clean)

Routes bloom like flowers
Python whispers to the web
Requests find their home


In [14]:
while attempts < max_attempts and not success:
    attempts += 1

    try:
        response_clean = generate_haiku_openrouter(topic)

        counts, is_perfect = evaluate_haiku(response_clean)

        if is_perfect:
            new_haiku = response_clean
            success = True
            print(f"  ✅ Success on attempt {attempts}: {counts}")
        elif attempts % 10 == 0:
            print(f"  Attempt {attempts}: {counts}")

    except Exception as e:
        print(f"  API error: {e}")

  Attempt 10: [6, 6, 5]
  Attempt 20: [6, 8, 5]
  Attempt 30: [7, 9, 6]


In [15]:
import string
import json
import os
from mlx_lm import load, generate
from mlx_lm.generate import make_sampler

/Users/vi/Code/mac-gemma-haiku/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
try:
    from gemmaiku import get_syllable_count_for_line as count_syllables_line
except ModuleNotFoundError:
    import sys
    import os
    sys.path.append(os.path.abspath("../../src"))
    from gemmaiku import get_syllable_count_for_line as count_syllables_line


In [17]:
def clean_haiku_response(text):
    lines = []
    for line in text.split('\n'):
        line_clean = line.strip().strip('*_-"\'#')
        if not line_clean:
            continue
        lower_line = line_clean.lower()
        if any(phrase in lower_line for phrase in [
            "here is", "haiku about", "following", "syllable count", "verify", 
            "sure, here", "ok, here", "perfect", "attempt", "topic", "lines have"
        ]):
            continue
        lines.append(line_clean)
    if len(lines) > 3:
        return "\n".join(lines[-3:])
    return "\n".join(lines)

In [52]:
# Load dataset
# Set to 'haikus_dataset.json' to fix leftovers, or 'haikus.json' to start fresh
dataset_file = 'haikus_dataset.json'

with open(dataset_file, 'r') as f:
    dataset = json.load(f)

# Find which ones need fixing
imperfect_indices = []
for idx, entry in enumerate(dataset):
    gpt_response = entry['conversations'][1]['value']
    _, is_perfect = evaluate_haiku(gpt_response)
    if not is_perfect:
        imperfect_indices.append(idx)

print(f"Found {len(imperfect_indices)} imperfect haikus out of {len(dataset)} in {dataset_file}")


Found 2 imperfect haikus out of 500 in haikus_dataset.json


In [53]:
# Fixer Configuration
USE_OPENROUTER = True  # Set to True to use OpenRouter (Claude), False to use local Gemma-3-1b-it

if len(imperfect_indices) > 0:
    if USE_OPENROUTER:
        print(f"Using OpenRouter ({OPENROUTER_MODEL}) for correction...")
    else:
        print("Loading local Gemma model for correction...")
        # Load model
        model, tokenizer = load("google/gemma-3-1b-it")
    
    # We can run in a loop
    for count, idx in enumerate(imperfect_indices, 1):
        entry = dataset[idx]

        print(f"---{idx}")
        
        topic = entry['conversations'][0]['value']
        print(f"\n[{count}/{len(imperfect_indices)}] Fixing topic: '{topic}'")
        
        # We loop until we get a perfect 5-7-5 haiku
        attempts = 0
        max_attempts = 1 if USE_OPENROUTER else 30
        success = False
        new_haiku = ""
        
        while attempts < max_attempts and not success:
            attempts += 1
            
            if USE_OPENROUTER:
                try:
                    response_clean = generate_haiku_openrouter(topic)
                    counts, is_perfect = evaluate_haiku(response_clean)
                    
                    if is_perfect:
                        new_haiku = response_clean
                        success = True
                        print(f"  ✅ Success on OpenRouter attempt {attempts}: {counts}")
                        print(f"  {new_haiku.replace('\n', ' / ')}")
                    else:
                        if attempts % 5 == 0:
                            print(f"  Attempt {attempts} count pattern: {counts} - Retrying...")
                except Exception as e:
                    print(f"  OpenRouter API error: {e}")
            else:
                # Vary temperature slightly across attempts to prevent repetitive patterns
                temp = 0.3 + 0.1 * (attempts % 6)
                sampler = make_sampler(temp=temp)
                
                # Use different prompt styles to guide the model if it gets stuck
                if attempts <= 10:
                    # Strong few-shot prompt to enforce format
                    prompt = (
                        f"<start_of_turn>user\nWrite a strictly 5-7-5 syllable haiku about: What is the square root of 16?<end_of_turn>\n"
                        f"<start_of_turn>model\nFour times four is math,\nSimple root of sixteen found,\nNumber stands alone.<end_of_turn>\n"
                        f"<start_of_turn>user\nWrite a strictly 5-7-5 syllable haiku about: {topic}<end_of_turn>\n"
                        f"<start_of_turn>model\n"
                    )
                elif attempts <= 20:
                    prompt = f"<start_of_turn>user\nWrite a strictly 5-7-5 syllable haiku about: {topic}. Double check that your lines have exactly 5, 7, and 5 syllables.<end_of_turn>\n<start_of_turn>model\n"
                else:
                    prompt = f"<start_of_turn>user\nYou are a precision 5-7-5 haiku writer. Write a haiku about: {topic}. Line 1 must be 5 syllables, line 2 must be 7, line 3 must be 5.<end_of_turn>\n<start_of_turn>model\n"
                    
                response = generate(model, tokenizer, prompt=prompt, sampler=sampler, max_tokens=64)
                
                if "<end_of_turn>" in response:
                    response = response.split("<end_of_turn>")[0]
                
                response_clean = clean_haiku_response(response)
                counts, is_perfect = evaluate_haiku(response_clean)
                
                if is_perfect:
                    new_haiku = response_clean
                    success = True
                    print(f"  ✅ Success on attempt {attempts}: {counts}")
                    print(f"  {new_haiku.replace('\n', ' / ')}")
                else:
                    if attempts % 10 == 0:
                        print(f"  Attempt {attempts} count pattern: {counts} - Retrying...")
        
        if success:
            dataset[idx]['conversations'][1]['value'] = new_haiku
        else:
            print(f"  ⚠️ Warning: Failed to generate perfect haiku for '{topic}' after {max_attempts} attempts. Keeping original.")
            
    # Save the updated dataset
    # Note: we save back to haikus_dataset.json
    with open('haikus_dataset.json', 'w') as f:
        json.dump(dataset, f, indent=2)
    print("\nDataset updated successfully!")
else:
    print("All haikus are already perfect!")


Using OpenRouter (anthropic/claude-sonnet-4.6) for correction...
---258

[1/2] Fixing topic: 'How do I make a pizza?'
  ⚠️ Warning: Failed to generate perfect haiku for 'How do I make a pizza?' after 1 attempts. Keeping original.
---266

[2/2] Fixing topic: 'Tell me about the Great Wall of China.'
  ⚠️ Warning: Failed to generate perfect haiku for 'Tell me about the Great Wall of China.' after 1 attempts. Keeping original.

Dataset updated successfully!


In [24]:
def manual_replace_haiku(topic, correct_haiku, filepath='haikus_dataset.json'):
    """
    Manually replace a haiku for a specific topic in the dataset file after verifying it is 5-7-5.
    """
    # Verify the haiku syllable count first
    counts, is_perfect = evaluate_haiku(correct_haiku)
    if not is_perfect:
        print(f"⚠️ Warning: The provided haiku is not 5-7-5. Syllable counts: {counts}")
        print("Please check the syllables and try again.")
        return False
    
    # Load dataset
    with open(filepath, 'r') as f:
        dataset = json.load(f)
        
    # Search for the topic in the dataset
    found = False
    for entry in dataset:
        if entry['conversations'][0]['value'].strip().lower() == topic.strip().lower():
            entry['conversations'][1]['value'] = correct_haiku.strip()
            found = True
            break
            
    if found:
        # Save updated dataset
        with open(filepath, 'w') as f:
            json.dump(dataset, f, indent=2)
        print(f"✅ Successfully replaced haiku for topic '{topic}' in {filepath}!")
        return True
    else:
        print(f"❌ Topic '{topic}' not found in the dataset.")
        return False

# Example usage:
# topic_to_replace = "Who wrote 'Romeo and Juliet'?"
# new_correct_haiku = """
# Bard of Avon's pen,
# Star-crossed lovers meet their end,
# Timeless tragedy.
# """.strip()
# manual_replace_haiku(topic_to_replace, new_correct_haiku)


In [41]:
topic_to_replace = "How do I use a knife?"

In [42]:
new_correct_haiku = """
Knife use hand
Hold grip sharp board slice food
Safe care hands cut
""".strip()

In [43]:
manual_replace_haiku(topic_to_replace, new_correct_haiku)

✅ Successfully replaced haiku for topic 'How do I use a knife?' in haikus_dataset.json!


True

In [40]:
for w in [
    "knife", "use", "hold", "grip", "cut",
    "slice", "board", "sharp", "safe", "care",
    "hand", "hands", "food"
]:
    print(w, count_syllables_line(w))

knife 2
use 2
hold 1
grip 1
cut 1
slice 2
board 1
sharp 1
safe 2
care 1
hand 1
hands 1
food 1


In [ ]:
print(count_syllables_line("Only own and one a"))

In [54]:
import json

with open("haikus_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data[:1] if isinstance(data, list) else list(data.keys())[:5])

<class 'list'>
[{'conversations': [{'from': 'human', 'value': 'What is the capital of France?'}, {'from': 'gpt', 'value': 'Paris holds the key,\nCity of lights, grand and bright,\nCapital stands proud.'}]}]


In [55]:
import json

with open("haikus_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data[258], indent=2, ensure_ascii=False))

{
  "conversations": [
    {
      "from": "human",
      "value": "How do I make a pizza?"
    },
    {
      "from": "gpt",
      "value": "Dough and sauce and cheese,\nFire makes the crust so crisp,\nSlice of Italy."
    }
  ]
}


In [57]:
def manual_replace_entry(
    id,
    question,
    haiku,
    filepath='haikus_dataset.json'
):
    """
    Replace an entire dataset entry (question + haiku) by index.
    """

    counts, is_perfect = evaluate_haiku(haiku)
    if not is_perfect:
        print(f"⚠️ Warning: The provided haiku is not 5-7-5. Syllable counts: {counts}")
        return False

    with open(filepath, 'r', encoding='utf-8') as f:
        dataset = json.load(f)

    if not (0 <= id < len(dataset)):
        print(f"❌ Invalid id: {id}")
        return False

    dataset[id] = {
        "conversations": [
            {
                "from": "human",
                "value": question.strip()
            },
            {
                "from": "gpt",
                "value": haiku.strip()
            }
        ]
    }

    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=2, ensure_ascii=False)

    print(f"✅ Replaced entry {id}")
    return True

In [60]:
manual_replace_entry(
    id=266,
    question="Why do leaves change color?",
    haiku="""
Autumn season cool
Harvest leaves wind gold red
Brown trees crisp fall leaf
""".strip()
)

✅ Replaced entry 266


True